In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
import pandas as pd
train=pd.read_csv('/content/gdrive/MyDrive/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/content/gdrive/MyDrive/house-prices-advanced-regression-techniques/test.csv')

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

label_encoder = LabelEncoder()

scaler = StandardScaler()

for col in train.columns:
    if train[col].dtype == 'object':
        if train[col].isin([True, False]).all():
            train[col] = train[col].map({True: 1, False: 0})
        else:
            train[col] = label_encoder.fit_transform(train[col])
    else:
        pass

for col in test.columns:
    if test[col].dtype == 'object':
        if test[col].isin([True, False]).all():
            test[col] = test[col].map({True: 1, False: 0})
        else:
            test[col] = label_encoder.fit_transform(test[col])


In [ ]:
correlation_matrix = train.corr()

correlation_with_saleprice = correlation_matrix['SalePrice'].sort_values(ascending=False)

top_50_correlated = correlation_with_saleprice[1:26] + correlation_with_saleprice[56:81]

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

# Select top 50 features
top_50_features = top_50_correlated.index.tolist()
X_train = train[top_50_features]
y_train = train['SalePrice']
X_test = test[top_50_features]

# Impute missing values - only numeric columns
imputer = SimpleImputer(strategy='median')

# Make sure that only numeric columns are passed to the imputer
X_train_imputed = imputer.fit_transform(X_train.select_dtypes(include=['number']))
X_test_imputed = imputer.transform(X_test.select_dtypes(include=['number']))

# Scale data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# Fit Lasso model
lasso = Lasso(alpha=1)
lasso.fit(X_train_scaled, y_train)

# Make predictions
y_pred = lasso.predict(X_test_scaled)

# Prepare predictions for output
predictions = list(zip(test['Id'], y_pred))

# Create and save predictions DataFrame
predictions_df = pd.DataFrame(predictions, columns=['Id', 'SalePrice'])
predictions_df.to_csv('predicted_prices.csv', index=False)

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.081e+11, tolerance: 9.208e+08
  model = cd_fast.enet_coordinate_descent(
